# LAB 4 — LangFuse: Observabilidade para Sistemas RAG Jurídicos

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 1 — Fundamentos  
**Duração estimada:** 20 minutos  
**Ambiente:** Jupyter Local no VS Code

---

## Objetivos de Aprendizagem
1. Criar conta e projeto no LangFuse
2. Configurar API Keys via arquivo `.env` local
3. Registrar o primeiro trace de uma chamada LLM simulada
4. Visualizar traces no dashboard do LangFuse
5. Entender por que observabilidade é crítica em sistemas RAG jurídicos

## Por que Observabilidade?

Em sistemas RAG para o setor jurídico, cada resposta gerada precisa ser auditável:  
quais documentos foram recuperados? Qual foi o prompt exato enviado ao LLM?  
O modelo seguiu as instruções? O LangFuse registra tudo isso automaticamente,  
permitindo auditoria, debugging e melhoria contínua do sistema.

## Pré-requisito
- Conta criada em https://cloud.langfuse.com (gratuita)
- Lab 2 e Lab 3 concluídos
- Ollama rodando (para o Trace com chamada real ao LLM)

## CONFIGURAÇÃO INICIAL — Criar Conta e Projeto no LangFuse

**Execute ANTES de rodar as células abaixo:**

1. Acesse **https://cloud.langfuse.com** e crie uma conta gratuita
2. Crie um novo projeto chamado **`mba-rag-direito`**
3. Vá em **Settings → API Keys → Create new API Key**
4. Copie a **Public Key** (começa com `pk-...`) e a **Secret Key** (começa com `sk-...`)
5. Adicione ao arquivo `~/mba-rag/.env`:
   ```
   LANGFUSE_PUBLIC_KEY=pk-lf-sua-chave-aqui
   LANGFUSE_SECRET_KEY=sk-lf-sua-chave-aqui
   LANGFUSE_HOST=https://cloud.langfuse.com
   ```

> **Para dados sigilosos:** Em tribunais e delegacias, use LangFuse self-hosted via Docker  
> para não enviar metadados para a nuvem. Consulte o ROTEIRO_INSTALACAO_FERRAMENTAS.md.

## 📦 CÉLULA 1 — Instalar e Configurar LangFuse

In [ ]:
# CÉLULA 1: Instalar LangFuse e carregar configuracoes
%pip install langfuse==2.36.1 python-dotenv --quiet

import os
from pathlib import Path

# Carrega variaveis do arquivo .env local (sem google.colab)
try:
    from dotenv import load_dotenv
    possiveis_env = [
        Path.home() / 'mba-rag' / '.env',
        Path('.env'),
        Path('..') / '.env',
    ]
    for env_path in possiveis_env:
        if env_path.exists():
            load_dotenv(env_path)
            print(f'Configuracoes carregadas de: {env_path.resolve()}')
            break
    else:
        print('Arquivo .env nao encontrado — usando variaveis de ambiente existentes')
except ImportError:
    pass

LANGFUSE_PUBLIC_KEY = os.environ.get('LANGFUSE_PUBLIC_KEY', '')
LANGFUSE_SECRET_KEY = os.environ.get('LANGFUSE_SECRET_KEY', '')
LANGFUSE_HOST = os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')

# Configura variaveis de ambiente para os frameworks
os.environ['LANGFUSE_PUBLIC_KEY'] = LANGFUSE_PUBLIC_KEY
os.environ['LANGFUSE_SECRET_KEY'] = LANGFUSE_SECRET_KEY
os.environ['LANGFUSE_HOST'] = LANGFUSE_HOST

def status_chave(chave):
    if chave and not chave.endswith('_AQUI') and len(chave) > 10:
        return f'Configurada ({chave[:8]}...)'
    return 'NAO CONFIGURADA — adicione ao arquivo .env'

print()
print('CONFIGURACAO LANGFUSE')
print('=' * 50)
print(f'Public Key: {status_chave(LANGFUSE_PUBLIC_KEY)}')
print(f'Secret Key: {status_chave(LANGFUSE_SECRET_KEY)}')
print(f'Host:       {LANGFUSE_HOST}')


## 📦 CÉLULA 2 — Criar Cliente LangFuse e Validar Conexão

In [ ]:
# Célula 2: Criar cliente LangFuse e validar conexão
from langfuse import Langfuse

# Cria cliente LangFuse
# O cliente usa automaticamente as variáveis de ambiente
langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST
)

# Valida as credenciais
try:
    langfuse.auth_check()
    print('✅ Conexão com LangFuse validada!')
    print(f'   Projeto: mba-rag-direito')
    print(f'   Host:    {LANGFUSE_HOST}')
except Exception as e:
    print(f'❌ Erro de autenticação: {e}')
    print('   Verifique se as API Keys estão corretas nos Segredos do Colab')

## 📦 CÉLULA 3 — Primeiro Trace Manual

**O que faz:** Registra manualmente o ciclo completo de uma chamada RAG: query → retrieval → LLM → resposta.

**Por que:** Entender a estrutura de um trace ajuda a monitorar sistemas RAG em produção.

In [ ]:
# Célula 3: Primeiro trace manual — Simulação de pipeline RAG
import time
import uuid

print('📊 REGISTRANDO PRIMEIRO TRACE NO LANGFUSE')
print('=' * 60)

# ─── 1. Cria um Trace (representa uma interação completa) ─────
trace = langfuse.trace(
    name='consulta-juridica-rag',           # Nome descritivo
    user_id='advogado-demo-001',             # ID do usuário (anonimizado)
    session_id=str(uuid.uuid4()),            # ID único da sessão
    metadata={
        'modulo': 'MBA RAG - Aula 1',
        'tipo_consulta': 'jurisprudencia',
        'tribunal_alvo': 'STJ',
        'ambiente': 'jupyter-local'
    },
    tags=['aula1', 'demo', 'fundamentos']
)

print(f'✅ Trace criado: {trace.id}')

# ─── 2. Span: Fase de Retrieval ───────────────────────────────
# Um span representa uma etapa específica dentro do trace
retrieval_span = trace.span(
    name='documento-retrieval',
    metadata={
        'indice': 'documentos_juridicos_aula1',
        'top_k': 3,
        'metodo': 'BM25 (lab1 - ainda sem embedding)'
    }
)

# Simula busca (em labs posteriores, este será o OpenSearch real)
time.sleep(0.3)  # Simula latência de busca

documentos_recuperados = [
    {'id': '1', 'titulo': 'Acórdão - Peculato - STJ', 'score': 0.92},
    {'id': '2', 'titulo': 'Habeas Corpus - Prisão Preventiva', 'score': 0.78},
    {'id': '5', 'titulo': 'Sentença - Legítima Defesa', 'score': 0.54}
]

retrieval_span.end(
    output=documentos_recuperados,
    status_message='3 documentos recuperados'
)
print(f'✅ Span de retrieval registrado ({len(documentos_recuperados)} docs)')

# ─── 3. Generation: Chamada ao LLM ───────────────────────────
# LangFuse.generation é especial — monitora custo, tokens e qualidade
contexto = '\n'.join([d['titulo'] for d in documentos_recuperados])
prompt_final = f"""
Baseado nos seguintes documentos jurídicos recuperados:
{contexto}

Responda: Qual a diferença entre peculato e estelionato no direito penal brasileiro?
"""

generation = trace.generation(
    name='llm-resposta-juridica',
    model='llama3.2:3b',
    model_parameters={
        'temperature': 0.2,
        'max_tokens': 300,
        'top_p': 0.9
    },
    input=prompt_final,
    metadata={'fase': 'geracao_resposta'}
)

# Simula geração (em produção, aqui estaria a chamada real ao vLLM)
time.sleep(0.5)

resposta_simulada = """
PECULATO (art. 312 CP): Crime próprio de servidor público que se apropria 
de dinheiro, valor ou bem público de que tem posse em razão do cargo.

ESTELIONATO (art. 171 CP): Crime comum (qualquer pessoa) que obtém 
vantagem ilícita mediante fraude, induzindo alguém em erro.

DISTINÇÃO PRINCIPAL: O peculato é crime funcional (somente servidor 
público pode praticar); o estelionato é crime comum. No peculato, 
o agente já tem posse do bem; no estelionato, induz a vítima a entregá-lo.
"""

generation.end(
    output=resposta_simulada,
    usage={
        'input': 87,   # Tokens do prompt
        'output': 142, # Tokens da resposta
        'total': 229   # Total
    }
)
print('✅ Generation registrada no LangFuse')

# ─── 4. Score: Avaliação da resposta ─────────────────────────
# Scores permitem avaliar qualidade das respostas ao longo do tempo
trace.score(
    name='relevancia',
    value=0.9,        # 0.0 a 1.0
    comment='Resposta correta e bem fundamentada para os documentos recuperados'
)

trace.score(
    name='precisao_juridica',
    value=0.95,
    comment='Artigos citados corretos (312 e 171 CP)'
)

# Envia tudo para o LangFuse
langfuse.flush()
print('\n✅ Trace completo enviado para o LangFuse!')
print(f'\n🌐 Visualize em: {LANGFUSE_HOST}')
print(f'   Projeto: mba-rag-direito → Traces')
print(f'   Trace ID: {trace.id}')

### 📤 Output Esperado:
```
✅ Trace criado: cma1b2c3d4e5f6...
✅ Span de retrieval registrado (3 docs)
✅ Generation registrada no LangFuse
✅ Trace completo enviado para o LangFuse!

🌐 Visualize em: https://cloud.langfuse.com
   Projeto: mba-rag-direito → Traces
```

## 📦 CÉLULA 4 — Decorator para Observabilidade Automática

**O que faz:** Demonstra o uso de `@observe` para registrar automaticamente funções Python no LangFuse.

**Por que:** Em sistemas reais, não queremos instrumentar manualmente cada chamada. O decorator `@observe` faz isso automaticamente.

In [ ]:
# Célula 4: Usando @observe para instrumentação automática
from langfuse.decorators import observe, langfuse_context

@observe(name='pipeline-rag-completo')
def pipeline_rag_simples(pergunta: str, top_k: int = 3) -> str:
    """
    Pipeline RAG simplificado com observabilidade automática.
    O decorator @observe registra automaticamente:
    - Input (pergunta + parâmetros)
    - Output (resposta)
    - Duração
    - Erros (se houver)
    """
    
    # Adiciona metadados ao trace atual
    langfuse_context.update_current_trace(
        user_id='usuario-demo',
        tags=['pipeline-rag', 'aula1-demo'],
        metadata={'pergunta_tipo': 'classificacao_criminal'}
    )
    
    # Etapa 1: Retrieval (simulado)
    documentos = etapa_retrieval(pergunta, top_k)
    
    # Etapa 2: Geração (simulada)
    resposta = etapa_geracao(pergunta, documentos)
    
    return resposta


@observe(name='etapa-retrieval')
def etapa_retrieval(pergunta: str, top_k: int) -> list:
    """Busca documentos relevantes — instrumentada automaticamente."""
    time.sleep(0.2)  # Simula latência
    # Em produção: busca real no OpenSearch/FAISS
    return [
        {'id': 'doc1', 'relevancia': 0.89, 'titulo': 'CP Art. 155 - Furto'},
        {'id': 'doc2', 'relevancia': 0.76, 'titulo': 'CP Art. 157 - Roubo'}
    ]


@observe(name='etapa-geracao')
def etapa_geracao(pergunta: str, documentos: list) -> str:
    """Gera resposta com LLM — instrumentada automaticamente."""
    time.sleep(0.3)  # Simula latência do LLM
    # Em produção: chamada real ao vLLM
    return f'Baseado em {len(documentos)} documentos: resposta jurídica gerada aqui.'


# Executa o pipeline — todos os spans são registrados automaticamente
print('🔄 Executando pipeline RAG com @observe...')
resultado = pipeline_rag_simples(
    pergunta='Qual é a diferença entre furto simples e roubo?',
    top_k=2
)

langfuse.flush()  # Garante envio de todos os eventos

print(f'✅ Pipeline executado!')
print(f'   Resposta: {resultado}')
print(f'\n🌐 Verifique no LangFuse: Traces → pipeline-rag-completo')

# Sumário do Lab 4
print('\n' + '=' * 60)
print('📋 CONCEITOS DEMONSTRADOS NESTE LAB:')
print('   ✅ trace()      — registro de uma interação completa')
print('   ✅ span()       — etapa específica (retrieval, preprocessing)')
print('   ✅ generation() — chamada ao LLM com tokens e modelo')
print('   ✅ score()      — avaliação de qualidade da resposta')
print('   ✅ @observe     — instrumentação automática de funções')

## Checklist de Validação do Lab 4

- [ ] Conta criada em cloud.langfuse.com
- [ ] API Keys configuradas no arquivo `.env`
- [ ] `langfuse.auth_check()` passou sem erros
- [ ] Trace criado e visível no dashboard LangFuse
- [ ] Span de retrieval registrado
- [ ] Generation com tokens registrada
- [ ] Pipeline com `@observe` executado e visível no LangFuse

**Pontuação:** 7/7 = Lab 4 completo

## Próximo Passo

Continue para o **LAB5 — Embeddings BGE-M3 + UMAP** para explorar  
visualização e comparação de modelos de embedding para o domínio jurídico.
